#1  Setup Dependency

In [1]:
# Jalankan ini di sel baru untuk memastikan library terpasang dengan benar
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" --no-deps
!pip install --no-deps "xformers<0.0.29" "trl<0.14.0" "peft" "accelerate" "bitsandbytes" "trl"
!pip install unsloth_zoo
!pip install -qU langchain langchain-community langchain-core langchain-text-splitters langchain-huggingface chromadb pypdf
!pip install -q rank_bm25

import os
import uuid
import importlib.metadata

import sys
!{sys.executable} -m pip install -U ddgs

import re
import torch
import re
from langchain_community.tools import DuckDuckGoSearchRun
from google.colab import files
from google.colab import drive
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.stores import InMemoryByteStore
from langchain_community.retrievers import BM25Retriever

from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import ChatPromptTemplate

from sentence_transformers import CrossEncoder


from unsloth import FastLanguageModel
from transformers import pipeline, BitsAndBytesConfig

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-4izye7ev/unsloth_41ed6ae8536649daa465a7f989d3632c
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-4izye7ev/unsloth_41ed6ae8536649daa465a7f989d3632c
  Resolved https://github.com/unslothai/unsloth.git to commit 1798ebe3f30281978abdc7e409bd91a11a1faa8a
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached trl-0.24.0-py3-none-any.whl.metadata (11 kB)
Using cached trl-0.24.0-py3-none-any.whl (423 kB)
  Attempting uninstall: trl
    Found existing installation: trl 0.13.0
    Uninstalling trl-0.13.0:
      Successfully uninstalled trl-0.13.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 89.9 MB/s e

/tmp/ipykernel_3168/1096024088.py:21: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader
/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:144: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
# Hubungkan Drive
drive.mount('/content/drive')
FOLDER_PATH = "/content/drive/MyDrive/IDCamp/DokumenRAG"

Mounted at /content/drive


#2 Load PDF and MetaData Enrichment

In [3]:
print(f"\nMemuat seluruh dokumen PDF dari: {FOLDER_PATH}...")
loader = PyPDFDirectoryLoader(FOLDER_PATH)
docs = loader.load()

# =================================================================
# TAMBAHAN KODE UNTUK VERIFIKASI REVIEWER (Sesuai feedback IDCamp)
# =================================================================
# Mengambil daftar nama file unik dari metadata 'source' seluruh halaman
unique_files = set()
for doc in docs:
    file_path_asal = doc.metadata.get('source', '')
    if file_path_asal:
        unique_files.add(os.path.basename(file_path_asal))

print(f"[INFO] Berhasil memuat total {len(docs)} halaman dari {len(unique_files)} dokumen PDF.")
print("\n--- BUKTI EKSPLISIT DOKUMEN YANG DIMUAT (Validasi Reviewer) ---")
for idx, file_name in enumerate(sorted(unique_files), start=1):
    print(f"{idx}. {file_name}")
print("---------------------------------------------------------------\n")
# =================================================================

# Menggabungkan stabilitas PyPDFDirectoryLoader dengan ketajaman Regex
for i, doc in enumerate(docs):
    # Ambil nama file asli dari metadata bawaan PyPDF
    file_path_asal = doc.metadata.get('source', '')
    file_name = os.path.basename(file_path_asal)
    title = os.path.splitext(file_name)[0]

    # Ekstraksi Regex
    number_match = re.search(r"Nomor\s+(\d+)", title, flags=re.IGNORECASE)
    year_match = re.search(r"Tahun\s+(\d{4})", title, flags=re.IGNORECASE)

    # Enrichment metadata sesuai kriteria penilaian Skilled/Advanced
    doc.metadata["document_title"] = title
    doc.metadata["regulation_number"] = int(number_match.group(1)) if number_match else -1
    doc.metadata["regulation_year"] = int(year_match.group(1)) if year_match else -1
    doc.metadata["category"] = "Hukum Positif Indonesia"
    doc.metadata["chunk_id_awal"] = f"doc_{i}"

    # Format Sitasi otomatis (Syarat mutlak kriteria Advanced)
    page_num = doc.metadata.get("page", 0) + 1 # PyPDF page index dimulai dari 0
    doc.metadata["citation"] = f"Dokumen: {title}, Halaman: {page_num}"

print("[SUCCESS] Metadata Enrichment tingkat Advanced berhasil diaplikasikan!")
print("Contoh metadata pada halaman pertama:")
print(docs[0].metadata)


Memuat seluruh dokumen PDF dari: /content/drive/MyDrive/IDCamp/DokumenRAG...
[INFO] Berhasil memuat total 1949 halaman dari 4 dokumen PDF.

--- BUKTI EKSPLISIT DOKUMEN YANG DIMUAT (Validasi Reviewer) ---
1. PP Nomor 35 Tahun 2021.pdf
2. PP Nomor 5 Tahun 2021.pdf
3. PP Nomor 51 Tahun 2023.pdf
4. UU Nomor 6 Tahun 2023.pdf
---------------------------------------------------------------

[SUCCESS] Metadata Enrichment tingkat Advanced berhasil diaplikasikan!
Contoh metadata pada halaman pertama:
{'producer': '', 'creator': 'Canon', 'creationdate': '2021-02-18T15:54:05+07:00', 'moddate': '2021-02-18T16:07:05+07:00', 'source': '/content/drive/MyDrive/IDCamp/DokumenRAG/PP Nomor 35 Tahun 2021.pdf', 'total_pages': 56, 'page': 0, 'page_label': '1', 'document_title': 'PP Nomor 35 Tahun 2021', 'regulation_number': 35, 'regulation_year': 2021, 'category': 'Hukum Positif Indonesia', 'chunk_id_awal': 'doc_0', 'citation': 'Dokumen: PP Nomor 35 Tahun 2021, Halaman: 1'}


#3 Parenting-Child Chunking

In [4]:
# (Parent & Child) - Rasio Emas (1500/150)
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=150)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)

parent_docs = parent_splitter.split_documents(docs)

#4 Embedding Open-Source & ChormaDB

In [5]:
#Setup Embedding Open Source (HuggingFace BGE-M3)
print("Memuat model embedding (BGE-M3)...")
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")

Memuat model embedding (BGE-M3)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

#5 Membuat Vector Database Lokal (ChromaDB)

In [6]:
vectorstore = Chroma(
    collection_name="legal_docs_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_db_legal"
)
store = InMemoryByteStore()

/tmp/ipykernel_3168/3041398582.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


#6 Parent-Child Retriever

In [7]:
# 6. Pengganti MultiVectorRetriever (Custom Retriever yang Stabil)
class MyCustomRetriever:
    def __init__(self, vectorstore, docstore):
        self.vectorstore = vectorstore
        self.docstore = docstore

    def get_relevant_documents(self, query: str):
        # 1. Cari Child Chunks yang paling relevan (k=5)
        child_hits = self.vectorstore.similarity_search(query, k=5)

        # 2. Ambil ID Parent unik dari Child yang ditemukan
        parent_ids = list(set([doc.metadata.get("doc_id") for doc in child_hits]))

        # 3. Tarik Dokumen Parent asli dari InMemoryByteStore
        parent_docs = self.docstore.mget(parent_ids)

        # Filter None jika ada ID yang tidak ditemukan
        return [doc for doc in parent_docs if doc is not None]

# Inisialisasi retriever
retriever = MyCustomRetriever(vectorstore=vectorstore, docstore=store)

# --- Proses Ingestion Data ---
id_key = "doc_id"
doc_ids = [str(uuid.uuid4()) for _ in parent_docs] # Generate unique IDs for parent docs

child_docs = []
for i, p_doc in enumerate(parent_docs):
    _id = doc_ids[i]
    _sub_docs = child_splitter.split_documents([p_doc])
    for _doc in _sub_docs:
        _doc.metadata[id_key] = _id
    child_docs.extend(_sub_docs)

print("Memasukkan vektor ke dalam database...")

# Implement batching for adding documents to vectorstore
batch_size = 500
for i in range(0, len(child_docs), batch_size):
    batch = child_docs[i:i + batch_size]
    vectorstore.add_documents(batch)
    print(f"  Added batch {i//batch_size + 1}/{(len(child_docs) + batch_size - 1)//batch_size} ({len(batch)} documents)")

store.mset(list(zip(doc_ids, parent_docs)))

print(f"\n[SUCCESS] RAG siap! {len(parent_docs)} Parent dan {len(child_docs)} Child Chunks tertanam!")

Memasukkan vektor ke dalam database...
  Added batch 1/18 (500 documents)
  Added batch 2/18 (500 documents)
  Added batch 3/18 (500 documents)
  Added batch 4/18 (500 documents)
  Added batch 5/18 (500 documents)
  Added batch 6/18 (500 documents)
  Added batch 7/18 (500 documents)
  Added batch 8/18 (500 documents)
  Added batch 9/18 (500 documents)
  Added batch 10/18 (500 documents)
  Added batch 11/18 (500 documents)
  Added batch 12/18 (500 documents)
  Added batch 13/18 (500 documents)
  Added batch 14/18 (500 documents)
  Added batch 15/18 (500 documents)
  Added batch 16/18 (500 documents)
  Added batch 17/18 (500 documents)
  Added batch 18/18 (427 documents)

[SUCCESS] RAG siap! 2324 Parent dan 8927 Child Chunks tertanam!


In [8]:
# --- UJI COBA PENCARIAN VEKTOR (SANITY CHECK) ---
print("\n--- Menguji Pencarian Parent-Child ---")
query_test = "Berapa lama waktu istirahat mingguan untuk pekerja?"

# Menggunakan method .get_relevant_documents() sesuai kelas MyCustomRetriever kita
hasil_pencarian = retriever.get_relevant_documents(query_test)

print(f"Ditemukan {len(hasil_pencarian)} Parent Dokumen yang relevan.\n")

if len(hasil_pencarian) > 0:
    for i, doc in enumerate(hasil_pencarian, 1):
        print(f"--- Top {i} ---")
        # Membuktikan Metadata Enrichment & Sitasi dari Sel 3 bekerja!
        print(f"Sitasi: {doc.metadata.get('citation', 'Tidak ada sitasi')}")
        print(f"Teks: {doc.page_content[:200]}...\n")
else:
    print("Peringatan: Tidak ada dokumen yang ditemukan. Coba ubah kata kunci query Anda.")


--- Menguji Pencarian Parent-Child ---
Ditemukan 5 Parent Dokumen yang relevan.

--- Top 1 ---
Sitasi: Dokumen: UU Nomor 6 Tahun 2023, Halaman: 558
Teks: PRESIDEN
REPTI9LIK IHDONESIA
-548-
23. Ketentuan Pasal 77 diubah sehingga berbunyi sebagai
berikut:
Pasal 77
(1) Setiap Pengusaha wajib melaksanakan ketentuan
waktu kerja.
(21 Waktu kerja sebagaimana ...

--- Top 2 ---
Sitasi: Dokumen: PP Nomor 35 Tahun 2021, Halaman: 15
Teks: PRESiDEN
REPUBLIK INDONESiA
- 15-
b. 8 (delapan)jam 1 (satu) hari dan 40 (empat puluh)
ja- 1 (satu) minggu untuk 5 (lima) hari kerja
dalam 1 (satu) minggu.
(3) Ketentuan waktu kerja sebagaimana dimaks...

--- Top 3 ---
Sitasi: Dokumen: PP Nomor 35 Tahun 2021, Halaman: 19
Teks: PRESIDEN
REPUELIK INDONESIA
-L9-
a. untuk ja- kerja lembur pertama sebesar
1,5 (satu koma lima) kali Upah sejam; dan
b. untuk setiap ja- kerja lembur berikutnya,
sebesar 2 (dua) kali Upah sejam.
(2) P...

--- Top 4 ---
Sitasi: Dokumen: PP Nomor 35 Tahun 2021, Halaman: 3
Teks: PRESIDEN
REP

#7 Ensemble Retriever BM25 + Semantic

In [9]:
# 1.Setup BM25 Retriever
bm25_retriever = BM25Retriever.from_documents(parent_docs)
bm25_retriever.k = 5

# 2. Definisi Custom Ensemble yang "Hacker-Proof"
class CustomEnsembleRetriever:
    def __init__(self, retriever_list, weights):
        self.retriever_list = retriever_list
        self.weights = weights

    def get_relevant_documents(self, query: str):
        # Mengambil hasil dari semantik (retriever) dan keyword (bm25)
        semantic_docs = self.retriever_list[0].get_relevant_documents(query)
        keyword_docs = self.retriever_list[1].get_relevant_documents(query)

        # Gabungkan hasil (karena ini hybrid search sederhana)
        return semantic_docs + keyword_docs

# 4. Inisialisasi
ensemble_retriever = CustomEnsembleRetriever(
    retriever_list=[retriever, bm25_retriever],
    weights=[0.6, 0.4]
)

print("[SUCCESS] Ensemble Retriever (Hybrid Search) siap dan stabil!")

[SUCCESS] Ensemble Retriever (Hybrid Search) siap dan stabil!


In [10]:
def search_and_display(query, retriever_obj, top_k=5):
    print(f"\n--- Mencari jawaban untuk: '{query}' ---")

    # Memanggil retriever hybrid kita
    relevant_docs = retriever_obj.get_relevant_documents(query)

    # Menampilkan hasil
    for i, doc in enumerate(relevant_docs[:top_k], start=1):
        print(f"--- Dokumen {i} ---")
        print(f"Sitasi: {doc.metadata.get('citation', 'Sumber tidak ditemukan')}")
        print(f"Konten:\n{doc.page_content[:600]}...") # Dibatasi 600 karakter agar rapi
        print("-" * 50)

# --- UJI COBA DENGAN PERTANYAAN BARU ---
# Anda bisa mengganti pertanyaan ini kapan saja!
pertanyaan_baru = "Berapa jam batas maksimal kerja lembur dalam satu minggu?"

# Fix: Redefine the CustomEnsembleRetriever locally with the corrected method call
# This is a local fix for this cell's execution. The original class in cell _C8ho6vkAQPL
# should ideally be updated for a permanent fix across the notebook.
class CustomEnsembleRetriever_Fixed:
    def __init__(self, retriever_list, weights):
        self.retriever_list = retriever_list
        self.weights = weights

    def get_relevant_documents(self, query: str):
        semantic_docs = self.retriever_list[0].get_relevant_documents(query)
        # The BM25Retriever from langchain_community.retrievers uses .invoke()
        # instead of .get_relevant_documents().
        keyword_docs = self.retriever_list[1].invoke(query)
        return semantic_docs + keyword_docs

# Re-initialize the ensemble_retriever using the fixed class
ensemble_retriever_fixed = CustomEnsembleRetriever_Fixed(
    retriever_list=[retriever, bm25_retriever],
    weights=[0.6, 0.4]
)

search_and_display(pertanyaan_baru, ensemble_retriever_fixed)


--- Mencari jawaban untuk: 'Berapa jam batas maksimal kerja lembur dalam satu minggu?' ---
--- Dokumen 1 ---
Sitasi: Dokumen: PP Nomor 35 Tahun 2021, Halaman: 19
Konten:
PRESIDEN
REPUELIK INDONESIA
-L9-
a. untuk ja- kerja lembur pertama sebesar
1,5 (satu koma lima) kali Upah sejam; dan
b. untuk setiap ja- kerja lembur berikutnya,
sebesar 2 (dua) kali Upah sejam.
(2) Perusahaan yang mempekerjakan Pekerja/Buruh
sebagaimana dimaksud pada ayat (1) wajib membayar
Upah Kerja Lembur, apabila kerja lembur dilakukan
pada hari istirahat mingguan dan/atau hari libur
resmi untuk waktu kerja 6 (enam) hari kerja dan
40 (empat puluh) jam seminggu, dengan ketentuan:
a. perhitungan Upah Kerja Lembur dilaksanakan
sebagai berikut:
1. jam pertama sampai dengan jam ketujuh,
dibaya...
--------------------------------------------------
--- Dokumen 2 ---
Sitasi: Dokumen: PP Nomor 35 Tahun 2021, Halaman: 17
Konten:
PRESIDEN
REPUBLIK INDONESIA
-t7-
Bagian Ketiga
Waktu Kerja Lembur
Pasal 26
(1) Waktu Kerja Lemb

#8 Reranker Top-K dan Relevance Score

In [11]:
# Redefine CustomEnsembleRetriever to correctly handle invoke and get_relevant_documents
# This temporary fix is placed here because modifications are restricted to this cell.
class CustomEnsembleRetriever_Fixed:
    def __init__(self, retriever_list, weights):
        self.retriever_list = retriever_list
        self.weights = weights

    def get_relevant_documents(self, query: str):
        # Mengambil hasil dari semantik (retriever) dan keyword (bm25)
        # The first retriever (MyCustomRetriever) has get_relevant_documents
        semantic_docs = self.retriever_list[0].get_relevant_documents(query)
        # The second retriever (BM25Retriever) uses .invoke()
        keyword_docs = self.retriever_list[1].invoke(query)

        # Gabungkan hasil (karena ini hybrid search sederhana)
        return semantic_docs + keyword_docs

    # Add an invoke method for compatibility with Runnable interface, if expected
    def invoke(self, query: str):
        return self.get_relevant_documents(query)


class HybridRAGEngine:
    def __init__(self, ensemble_retriever, reranker_model_name='cross-encoder/ms-marco-MiniLM-L-6-v2'):
        self.retriever = ensemble_retriever
        self.reranker = CrossEncoder(reranker_model_name)

    def _fetch_from_retriever(self, query):
        """Helper untuk mencoba berbagai method pengambilan data"""
        # With CustomEnsembleRetriever_Fixed, invoke() should now work correctly
        return self.retriever.invoke(query)

    def run(self, query, filter_dict=None, top_k_rerank=3): # <-- Argumen filter_dict ditambahkan di sini
        # 1. Hybrid Retrieval
        docs = self._fetch_from_retriever(query)

        if not docs:
            return [], 0.0

        # 2. METADATA FILTERING SEBELUM RERANKING
        if filter_dict:
            docs = [doc for doc in docs if metadata_matches(doc.metadata, filter_dict)]
            # Jika setelah difilter dokumennya habis, kembalikan list kosong
            if not docs:
                return [], 0.0

        # 3. Reranking (Hanya menilai dokumen yang sudah lolos filter tahun/UU)
        pairs = [(query, doc.page_content) for doc in docs]
        scores = self.reranker.predict(pairs)

        # 4. Sorting & Pemotongan
        scored_docs = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
        final_docs = [doc for doc, score in scored_docs[:top_k_rerank]]
        top_score = float(scored_docs[0][1])

        return final_docs, top_score

# --- Inisialisasi Ulang ---
# Re-initialize ensemble_retriever using the fixed class
ensemble_retriever_fixed = CustomEnsembleRetriever_Fixed(
    retriever_list=[retriever, bm25_retriever],
    weights=[0.6, 0.4]
)

# Initialize rag_engine with the fixed ensemble retriever
rag_engine = HybridRAGEngine(ensemble_retriever_fixed)

# --- Eksekusi ---
query = "Berapa jam batas maksimal kerja lembur dalam satu minggu?"
final_docs, top_score = rag_engine.run(query)

# --- Output ---
print(f"Relevance Score Top-1: {top_score:.4f}\n")
for i, doc in enumerate(final_docs, start=1):
    print(f"--- Dokumen {i} ---")
    print(f"Sitasi: {doc.metadata.get('citation', 'Sumber tidak ditemukan')}")
    print(f"Konten: {doc.page_content[:600]}...")
    print("-" * 50)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Relevance Score Top-1: 5.0529

--- Dokumen 1 ---
Sitasi: Dokumen: PP Nomor 35 Tahun 2021, Halaman: 17
Konten: PRESIDEN
REPUBLIK INDONESIA
-t7-
Bagian Ketiga
Waktu Kerja Lembur
Pasal 26
(1) Waktu Kerja Lembur hanya dapat dilakukan paling
lama 4 (empat)jam dalam 1 (satu) hari dan 18 (delapan
belas) jam dalam 1 (satu) minggu.
(2) Ketentuan Waktu Kerja Lembur sebagaimana
dimaksud pada ayat (1) tidak termasuk kerja lembur
yang dilakukan pada waktu istirahat mingguan
dan/atau hari libur resmi.
Pasal 27
(1) Pengusaha yang mempekerjakan Pekerja/Buruh
melebihi waktu kerja sebagaimana dimaksud dalam
Pasal 2l ayat (2), wajib membayar Upah Kerja Lembur.
(21 Kewajiban membayar Upah Kerja Lembur dikecualikan
bagi P...
--------------------------------------------------
--- Dokumen 2 ---
Sitasi: Dokumen: PP Nomor 35 Tahun 2021, Halaman: 17
Konten: PRESIDEN
REPUBLIK INDONESIA
-t7-
Bagian Ketiga
Waktu Kerja Lembur
Pasal 26
(1) Waktu Kerja Lembur hanya dapat dilakukan paling
lama 4 (empat)jam dalam 1 (sa

#9  Load Model Fine-Tuning

In [13]:
# 1. Konfigurasi Kuantisasi yang lebih presisi
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "ariefw666/qwen-finetuned-legal-16bit-model-1",
    max_seq_length = 2048,
    load_in_4bit = True,
    dtype = torch.bfloat16,
)

# 2. Setup Inference
FastLanguageModel.for_inference(model)

# 3. Text Generation Pipeline (Penting untuk kestabilan)
text_gen = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    do_sample=False,
    repetition_penalty=1.25,
    max_new_tokens=500,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id,
)

# 4. Bungkus ke LLM LangChain
llm = HuggingFacePipeline(pipeline=text_gen)

print("[SUCCESS] LLM Pipeline siap digunakan untuk RAG!")

==((====))==  Unsloth 2026.5.10: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Device does not support bfloat16. Will change to float16.


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[SUCCESS] LLM Pipeline siap digunakan untuk RAG!


#10 Prompt RAG dengan Sitasi

In [14]:
# Versi Refactor: Struktur Sitasi Pro + Qwen Format
prompt_template = """
<|im_start|>system
Anda adalah asisten ahli hukum Indonesia yang presisi.
Instruksi:
1. Jawab pertanyaan pengguna secara LANGSUNG dan RINGKAS berdasarkan KONTEKS.
2. Jangan menjelaskan pasal yang tidak relevan dengan pertanyaan.
3. Wajib mencantumkan sitasi [S1], [S2] untuk setiap klaim.
4. Jika informasi tidak ditemukan, katakan "Informasi tidak ditemukan".
5. Akhiri jawaban dengan bagian "Sumber:" yang merinci sitasi tersebut.

KONTEKS:
{context}
<|im_end|>
<|im_start|>user
{question}
<|im_end|>
<|im_start|>assistant
"""
def ask_rag_professional(query):
    # a. Tarik dokumen relevan
    final_docs, top_score = rag_engine.run(query)

    # b. Susun konteks untuk LLM
    context_text = ""
    for i, doc in enumerate(final_docs, 1):
        context_text += f"[S{i}] {doc.metadata.get('citation')}:\n{doc.page_content}\n\n"

    # c. Eksekusi LLM dengan prompt yang memerintahkan penggunaan format sitasi [S1]
    messages = [
        {"role": "system", "content": "Anda adalah asisten ahli hukum. Jawab ringkas, gunakan [S1], [S2] untuk sitasi klaim, dan jangan tulis bagian 'Sumber' di dalam jawaban LLM karena akan saya buat otomatis."},
        {"role": "user", "content": f"KONTEKS:\n{context_text}\n\nPERTANYAAN:\n{query}"}
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    response = llm.invoke(prompt)

    # Bersihkan output (menghapus prompt leakage)
    if "<|im_start|>assistant" in response:
        response = response.split("<|im_start|>assistant")[-1].strip()

    # d. Buat daftar sumber secara otomatis dari final_docs
    sumber_list = []
    for doc in final_docs:
        sumber_list.append(f"* {doc.metadata.get('citation', 'Sumber tidak ditemukan')}")

    # Unikkan daftar sumber (menghapus duplikat jika ada)
    unique_sumber = list(dict.fromkeys(sumber_list))

    # Gabungkan jawaban dan daftar sumber
    final_output = f"{response}\n\n**Sumber:**\n" + "\n".join(unique_sumber)

    return final_output

# Eksekusi
jawaban = ask_rag_professional("Berapa jam batas maksimal kerja lembur dalam satu minggu?")
print(jawaban)

Passing `generation_config` together with generation-related arguments=({'cache_implementation', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention m

Dari dokumen tersebut diperoleh bahwa setelah menyelesaikan tugasnya selama 4 jam per tanggal, seseorang bisa melakukan upaya tambahan terbatas selama 18 jam dalam seminggunya.

Jadi, jika Anda bekerja 4 jam pertama harinya, itu artinya Anda sudah melewati batasan atas jumlah jam kerja lembur dalam satu minggu. Namun, tetap ingatlah bahwa ini masih belum mencakup semua jenis kegiatan lain seperti makan siang, minuman, maupun aktivitas non-kerja lainnya. Jika ingin menggunakan waktu lembur lagi, pastikan Anda telah menyampaikan permintaan kepada managemenet tentang hal-hal apa saja yang membuat Anda butuhkan waktu extra. Selain itu, penting juga untuk dicatat bahwa beberapa posisi memiliki aturan sendiri tentang waktu lembur mereka, oleh karena itu baca informasi lengkap tentang situasi individu Anda agar menjaga dirimu aman saat bekerja ekstra.

**Sumber:**
* Dokumen: PP Nomor 35 Tahun 2021, Halaman: 17
* Dokumen: UU Nomor 6 Tahun 2023, Halaman: 558


#11 Metadata Filtering

In [26]:
def check_metadata(metadata_info, filter_rule):
    if filter_rule is None:
        return True

    if "$and" in filter_rule:
        return all(
            check_metadata(metadata_info, condition)
            for condition in filter_rule["$and"]
        )

    for field_name, target_value in filter_rule.items():
        if isinstance(target_value, dict):
            target_value = target_value.get("$eq", target_value)

        if metadata_info.get(field_name) != target_value:
            return False

    return True

#
def build_metadata_filter(query):
    normalized_query = query.lower()
    filter_conditions = []

    regulation_patterns = [
        ("pp 35", "PP Nomor 35 Tahun 2021"),
        ("pp nomor 35", "PP Nomor 35 Tahun 2021"),
        ("pp 5", "PP Nomor 5 Tahun 2021"),
        ("pp 51", "PP Nomor 51 Tahun 2023"),
        ("uu 6", "UU Nomor 6 Tahun 2023"),
        ("uu nomor 6", "UU Nomor 6 Tahun 2023"),
    ]

    for keyword, regulation_name in regulation_patterns:
        if keyword in normalized_query:
            filter_conditions.append(
                {"document_title": regulation_name}
            )
            break

    if not filter_conditions:
        return None

    if len(filter_conditions) == 1:
        return filter_conditions[0]

    return {"$and": filter_conditions}


def ask_rag_hybrid(query):
    filter_rule = build_metadata_filter(query)
    print(f"Filter terdeteksi: {filter_rule if filter_rule else 'Tidak ada (Global Search)'}")

    try:
        final_docs, top_score = rag_engine.run(query, filter_dict=filter_rule)
    except Exception as e:
        print(f"Peringatan: Gagal memfilter. Error: {e}")
        final_docs, top_score = rag_engine.run(query)

    if not final_docs:
        return "Informasi tidak ditemukan dalam dokumen hukum yang sesuai."

    context_text = ""
    for i, doc in enumerate(final_docs, 1):
        context_text += f"[S{i}] {doc.metadata.get('citation', 'Sumber Tidak Diketahui')}:\n{doc.page_content}\n\n"

    messages = [
        {"role": "system", "content": "Anda adalah asisten ahli hukum. Jawab ringkas, gunakan [S1], [S2] untuk sitasi klaim, dan JANGAN tulis bagian 'Sumber' di dalam jawaban LLM."},
        {"role": "user", "content": f"KONTEKS:\n{context_text}\n\nPERTANYAAN:\n{query}"}
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    response = llm.invoke(prompt)

    if "<|im_start|>assistant" in response:
        response = response.split("<|im_start|>assistant")[-1].strip()

    sumber_list = [f"* {doc.metadata.get('citation', 'Sumber tidak ditemukan')}" for doc in final_docs]
    unique_sumber = list(dict.fromkeys(sumber_list))

    final_output = f"{response}\n\n**Sumber:**\n" + "\n".join(unique_sumber)
    return final_output

# --- TEST ---
pertanyaan = "Apa aturan uang lembur menurut PP Nomor 35 Tahun 2021?"
hasil = ask_rag_hybrid(pertanyaan)

print("\n" + "="*50)
print("HASIL RAG HYBRID:")
print("="*50)
print(hasil)

Both `max_new_tokens` (=500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Filter terdeteksi: {'document_title': 'PP Nomor 35 Tahun 2021'}

HASIL RAG HYBRID:
PP Nomor 35 Tahun 2021 memberi penggantian Ulang Lebih Dari Normal kepada pekerja/buru jika mereka bekerja lebih banyak hari daripada jumlah harinya normal.

Untuk informasi lengkapnya, silahkan lihat dokumen tersebut secara langsung.

**Sumber:**
* Dokumen: PP Nomor 35 Tahun 2021, Halaman: 1
* Dokumen: PP Nomor 35 Tahun 2021, Halaman: 41


In [18]:
contoh_docs = rag_engine.retriever.invoke("uang lembur")

if contoh_docs:
    print("STRUKTUR METADATA DATABASE ANDA:")
    print("-" * 50)
    print(contoh_docs[0].metadata)
    print("-" * 50)
else:
    print("Database kosong atau retriever gagal mengambil data.")

STRUKTUR METADATA DATABASE ANDA:
--------------------------------------------------
{'producer': '', 'creator': 'Canon', 'creationdate': '2021-02-18T15:54:05+07:00', 'moddate': '2021-02-18T16:07:05+07:00', 'source': '/content/drive/MyDrive/IDCamp/DokumenRAG/PP Nomor 35 Tahun 2021.pdf', 'total_pages': 56, 'page': 18, 'page_label': '19', 'document_title': 'PP Nomor 35 Tahun 2021', 'regulation_number': 35, 'regulation_year': 2021, 'category': 'Hukum Positif Indonesia', 'chunk_id_awal': 'doc_18', 'citation': 'Dokumen: PP Nomor 35 Tahun 2021, Halaman: 19'}
--------------------------------------------------


#12 Hypothetical Document Embedding dan Advanced RAG

In [27]:
def generate_hypothetical_answer(query):

    messages = [
        {"role": "system", "content": "Anda adalah pakar hukum Indonesia. Berikan ringkasan perkiraan jawaban (hipotesis) untuk pertanyaan hukum berikut dalam 2-3 kalimat saja. Jangan gunakan sitasi."},
        {"role": "user", "content": query}
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    response = llm.invoke(prompt)

    if "<|im_start|>assistant" in response:
        response = response.split("<|im_start|>assistant")[-1].strip()

    return response



In [28]:
# Test singkat HyDE
print(generate_hypothetical_answer("Apa aturan potong gaji?"))

Both `max_new_tokens` (=500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Aturannya melibatkan penggunaan sistem yang memungkirkan pembagian jumlah total uang tetap menjadi sebagi antara karyawan dan perusahaan, dengan setiap anggota keluarga diberlakukan persentase tertentu dari sisa pendapatan bersih atau pajaknya.

Dalam hal ini, jika suatu organisasi memiliki satu orang sebagai pemilik utama dan dua anak di bawah usia sepenuh tahun, maka mereka akan menerima separuh porsi atas semua keuntungan selain dividen. Namun, ketika ada lebih banyak wali lain seperti istri, saudari laki-laki, ataupun temen kerja, maka nilai bagian tersebut dapat dibagi sesuai kesepakaman para peserta. Atau bisa juga digunakan secara eksklusif oleh salah satu individu tergantung pada undang-undang lokal maupun regulasinya sendiri.
Jadi, penting bagi Anda mengetahui bahwa tujuan dasarnya adalah mengatur distribusi hasil bisnis agar tidak merugikan siapa pun tanpanyeganelemen legalitas. Ini bukanlah masalah umum karena sering kali disesuaikannya dengan kondisi lokalisasi serta prinsi

#13 HyDE, Conditional Fallback DuckDuckGO, & Inference





In [29]:
web_search = DuckDuckGoSearchRun()

def ask_rag_advanced(query, use_hyde=False):
    # 1. RETRIEVAL & FILTERING
    metadata_filter = build_metadata_filter(query)

    try:
        final_docs, top_score = rag_engine.run(query, filter_dict=metadata_filter)
    except Exception:
        final_docs, top_score = rag_engine.run(query)

    if not final_docs:
        return do_web_search(query)

    context_text = ""
    for i, doc in enumerate(final_docs, 1):
        context_text += f"[S{i}] {doc.metadata.get('citation', 'Sumber Tidak Diketahui')}:\n{doc.page_content}\n\n"

    clean_query = re.sub(r'[^\w\s]', ' ', query).lower()
    context_lower = context_text.lower()

    stop_words = {
        "apa", "bagaimana", "aturan", "untuk", "di", "dalam", "dan", "yang",
        "menurut", "tahun", "indonesia", "terkait", "tentang", "pada", "ke",
        "dari", "ini", "itu", "apakah", "berapa", "saja", "sebutkan", "bagi",
        "sudah", "utama", "atau", "dengan", "mengenai", "lama", "berdasarkan",
        "terbaru", "saat", "ini", "ketentuan", "saja", "adalah", "karena",
        "serta", "yaitu", "hal", "tersebut", "bila", "jika", "agar", "supaya",
        "bisa", "dapat", "boleh", "diperbolehkan", "maksimal", "minimal", "paling",
        "mengatur", "sebagai", "secara", "ada", "nomor", "alasan", "dibutuhkan",
        "undang", "pemerintah", "peraturan", "undang-undang"
    }

    query_keywords = [w for w in clean_query.split() if w not in stop_words and not w.isdigit()]
    missing_keywords = [w for w in query_keywords if w not in context_lower]


    if len(missing_keywords) >= max(1, len(query_keywords) / 3):
        print(f"=> GUARDRAIL AKTIF: Kata {missing_keywords} tidak ada di dokumen lokal.")
        return do_web_search(query)


    messages = [
        {
            "role": "system",
            "content": "Anda adalah mesin pembaca dokumen. Anda tidak memiliki pengetahuan di luar dokumen yang diberikan."
        },
        {
            "role": "user",
            "content": (
                f"DOKUMEN:\n{context_text}\n\n"
                f"PERTANYAAN:\n{query}\n\n"
                "INSTRUKSI MUTLAK: Jika DOKUMEN di atas tidak berisi jawaban untuk pertanyaan ini, "
                "Anda DILARANG KERAS mengarang jawaban. Anda WAJIB membalas dengan satu kata saja: KOSONG."
            )
        }
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    response = llm.invoke(prompt)

    if "<|im_start|>assistant" in response:
        response = response.split("<|im_start|>assistant")[-1].strip()

    response = response.replace("[END_OF_TEXT]", "").replace("<|im_end|>", "").strip()

    if "KOSONG" in response.upper() or len(response.split()) < 3:
        print("=> LLM menolak berhalusinasi. Mengalihkan ke internet...")
        return do_web_search(query)

    sumber_list = [f"* {doc.metadata.get('citation', 'Sumber tidak ditemukan')}" for doc in final_docs]
    unique_sumber = list(dict.fromkeys(sumber_list))

    final_output = f"{response}\n\n**Sumber Lokal:**\n" + "\n".join(unique_sumber)
    return final_output

def do_web_search(query):
    try:
        web_query = f"Aturan hukum Indonesia terbaru: {query}"
        web_result = web_search.run(web_query)
        return f"**[🌐 PENCARIAN WEB AKTIF - Database Lokal Tidak Relevan]**\nBerikut adalah hasil dari internet:\n\n{web_result}"
    except Exception:
        return "Maaf, informasi tidak ditemukan di lokal dan pencarian web gagal."

#14 Interface Interactive Python Loop

In [30]:
def interactive_chat():
    print("="*60)
    print("⚖️ SISTEM ASISTEN HUKUM AI (RAG + WEB FALLBACK) AKTIF ⚖️")
    print("Ketik 'keluar', 'exit', atau 'quit' untuk mengakhiri sesi.")
    print("="*60)

    while True:
        user_query = input("\n🧑‍💻 Anda: ")

        # Deteksi perintah keluar
        if user_query.lower() in ['keluar', 'exit', 'quit']:
            print("\nSesi berakhir. Terima kasih!")
            break

        # Deteksi jika input kosong
        if not user_query.strip():
            print("Pertanyaan tidak boleh kosong.")
            continue

        print("🤖 Sistem sedang mencari referensi...")

        # Panggil fungsi RAG terbaik
        jawaban = ask_rag_advanced(user_query, use_hyde=False)

        print("\n" + "="*50)
        print(f"🤖 Asisten Hukum:\n{jawaban}")
        print("="*50)



In [31]:
# Uncomment baris di bawah ini untuk memulai chat!
interactive_chat()

⚖️ SISTEM ASISTEN HUKUM AI (RAG + WEB FALLBACK) AKTIF ⚖️
Ketik 'keluar', 'exit', atau 'quit' untuk mengakhiri sesi.

🧑‍💻 Anda: Bagaimana perhitungan upah kerja lembur pada hari istirahat mingguan menurut PP Nomor 35 Tahun 2021?


Both `max_new_tokens` (=500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 Sistem sedang mencari referensi...

🤖 Asisten Hukum:
Untuk menjawab pertanyaan tersebut, mari kita lihat bagaimana perhitungan upah kerja lembur terdiri dari dua langkah:

Langkah Pertama - Mengidentifikasi jumlah jam lembur.

Kerja lembur akan dipotong dari total jam kerja harian selama periode waktu kerja. Misalkan sebuah petugas bekerja 8 jam sehari normal, tetapi dia melakukan tugas tambahan selama beberapa jam lainnya. Ini bisa menjadi contoh seperti saat ia sedang merakit mobil sambil juga menyortir barang bekas ke tempat penyimpanan baru. Di situ, orang itu telah menciptakan nilai ekonomis positif karena membuat sesuatu yang belum tersedia sebelumnya. Namun, hal-hal seperti ini sering disebut "work in progress" sehingga sulit untuk ditandai akurat oleh manufaktur.

Berdasarkan informasi tentang jenis pekerjaan mereka sendiri serta kondisi industri umum, para ahli laboratorium biasanya menggunakan metrik seperti anggaran biaya produksi, potensi laba rugi, dan efektivitas operat

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Both `max_new_tokens` (=500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 Sistem sedang mencari referensi...

🤖 Asisten Hukum:
Tidak ada informasi tentang ketentuan waktu istirahat panjang yang harus dipertimbangkan oleh pengusaha dari konteks tersebut. Namun, pasal-paselannya menyediakan beberapa opsi waktu istirahat seperti:

* Istirahat antara jam kerja - minimal ½ jam setelah bekerja selama 4 jam terus menerus

* Istirahat mingguan 1 hari untuk 6 hari kerja dalam seminggu

Karena itu, jika sebuah organisasi hanya menjadwalkan dua jenis waktu istirahat sederhana tanpa mencantumkan waktu istirahat panjang, maka mereka akan tetap bertindih saat melakukan tugas-tugas rutinitas biasa. Tetapi apabila organisasi ingin membuat pendekatan yang lebih kompleks, misalnya dengan merekrut orang baru, merencanakan acara besar, ataupun berevolusi operasional bisnis, kemungkinan akan dibuat kesepakatan waktu istirahat panjang dengan para pegawai. Untuk mendapatkan detail lengkap tentang apa yang disyahkan oleh undangundiklat negri Indonesia serta persyaratan pelacaran,

...

🤖 Sistem sedang mencari referensi...

🤖 Asisten Hukum:
Formula penghitungan Upah minimum meliberalisa perbandingannya secara bertahap sesuai dengan tata kelola pendidikan tinggi.

[Formulas]
Ummin(t+1)= Ummin(t)+ [f(lagi)+(pejelasa -o)] * ummin[t]

Di mana:

* Ummin(t): Uptake Minimum saat ini
* f(lagi): Faktor Pertambangan Terakhir
* pejelasaa: Rasio Pembelian Sekitar
* O: Variabel Eksekutif Tingkat
* Ummin(t+1): Uptake Minimum selanjutnya 

Rumusan ini akan memberitahu kita cara mendapatkan formulir pengambilan Upah minimal baru setelah faktor-faktornya dipengaruhi oleh indikator-indikator seperti pertumbuhan ekonomi, inflasi, dan indeks tertentu. Rumus ini didesain untuk membantu meningkatkan persyaratan gaji industri tanpa merugikan buruh maupun bisnis.

**Sumber Lokal:**
* Dokumen: PP Nomor 51 Tahun 2023, Halaman: 17
* Dokumen: PP Nomor 51 Tahun 2023, Halaman: 3

🧑‍💻 Anda: Sebutkan apa saja ruang lingkup perizinan berusaha berbasis risiko menurut PP Nomor 5 Tahun 2021!


Both `max_new_tokens` (=500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 Sistem sedang mencari referensi...

🤖 Asisten Hukum:
Ruang Lingkup Perizinan Berusaha Berbasis Risiko:

Perusahaan Mikro atau Usia Menawan (UM-K), bisnis mikro maupun umusawati.
Usaha Besar.

Sistem dan Transaksi Elektornik,
Telepon, Televisi, Pertanian, Industri Kimia, Sains Terapan, Teknik Pengolahan Air, Transportasi Umum, Pelacaran Olahraga, Bisnis Properti, Bimbingan Belajar Online, Layanan Internet, Media Publisih, Akses Informasi, Rekreasi, Hukuman Ekonomi, Kejahilan Eksperimen, Pembayaran Digital, Laporan Akhir, Pendidikan Sekolah Luar Biasa, Komputer, Software, Data Center, Informatika, Manajemen Inventaris, Distribusi Barang, Logistik, Konstruksi Bangunan, Desain Arsitektural, Real Estate Brokerage, Investasi Emas, Trading Saham, Asuransi Jiwa, Insinyuren Produk, Produksi Obat, Farmasi, Biologi Genetika, Ilmu Manusia, Laboratorium Praktik Medikal, Peternakan Anjing, Kelamin Ayam, Perlindungan Tanaman Buah-Buang, Gizi Sehat, Otoritas Moneter Nasional, Bank Dunia, Badai Globa

Both `max_new_tokens` (=500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 Sistem sedang mencari referensi...

🤖 Asisten Hukum:
"Outsourcing," artinya secara sederhana, merupakan proses menebus tugas-tugas tertentu dari organisasi mereka kepada vendor atau bisnis asosiasi, umumnya untuk meningkatkan efektivitas, produktivitas, dan biaya komersial selain itu juga dapat membantu menjaga kontrol pasar serta menciptakan nilai tambah bagi produk akhir. Ini sering digunakan dalam industri seperti telekomunikasi, transportasi, layanan media sosial, hiburan, dll., untuk menyediakan solusi lebih fleksibel dan murah tanpa harus bertambah modal investasi.

**Sumber Lokal:**
* Dokumen: UU Nomor 6 Tahun 2023, Halaman: 589
* Dokumen: UU Nomor 6 Tahun 2023, Halaman: 841
* Dokumen: UU Nomor 6 Tahun 2023, Halaman: 638

🧑‍💻 Anda: exit

Sesi berakhir. Terima kasih!


In [32]:
# Uncomment baris di bawah ini untuk memulai chat!
interactive_chat()

⚖️ SISTEM ASISTEN HUKUM AI (RAG + WEB FALLBACK) AKTIF ⚖️
Ketik 'keluar', 'exit', atau 'quit' untuk mengakhiri sesi.

🧑‍💻 Anda: Bagaimana putusan Mahkamah Konstitusi (MK) terbaru mengenai gugatan uji materiil UU Cipta Kerja?


Both `max_new_tokens` (=500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 Sistem sedang mencari referensi...

🤖 Asisten Hukum:
Putusan MK nomor 91/PUU-XIIX II OOOO menyatakan bahwa undang-undang nomor 11 Tahun 2020 tentang cipta kerja bertentangan dengan konstansi negara Indonesia pada tanggal 1 Juli 1945 dan tidak mampu memberikan hak-hak kepada orang-orang tanpa adanya kesepakatan atau ketertiban tertulis.

Sumber : https://www.mkn.go.id/news/detail/id/nama-negara-indo-penuh-konstruktivitas-dari-mhs-tim-bidaya-selasa-2-october-2020?utm_source=google&utm_medium=cpc&gclid=CNCzNfZvYsACFQbEeWqMxHtAQ&smtol=en-US#content_1

**Sumber Lokal:**
* Dokumen: UU Nomor 6 Tahun 2023, Halaman: 751
* Dokumen: UU Nomor 6 Tahun 2023, Halaman: 8

🧑‍💻 Anda: Berapa nominal pasti Upah Minimum Kabupaten (UMK) Sukabumi untuk tahun ini?
🤖 Sistem sedang mencari referensi...
=> GUARDRAIL AKTIF: Kata ['nominal', 'pasti', 'sukabumi'] tidak ada di dokumen lokal.

🤖 Asisten Hukum:
**[🌐 PENCARIAN WEB AKTIF - Database Lokal Tidak Relevan]**
Berikut adalah hasil dari internet:

Dec 22, 20

Both `max_new_tokens` (=500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 Sistem sedang mencari referensi...

🤖 Asisten Hukum:
Sebagai AI model bahasa alami saya tidak bisa memberitahu informasi aktual seperti tingkat pajak tertinggi saat ini di negara-negara lain tanpa data langsung dari situs web resmi mereka.

Untuk mendapatkan informasi terupdate tentang pajak tertinggi di Indonesia, silahkan kunjungi website resminya - https://www.bpn.go.id/. Di sini anda akan mampu mencari tahu apa itu pajak pertambahan nilai barang dan juga bagaimana cara kerja sistem pajak serta kemungkinan ada promosi hingga tanggal 31 Desember tahun depan. Selamat menjelajahi!

**Sumber Lokal:**
* Dokumen: UU Nomor 6 Tahun 2023, Halaman: 637
* Dokumen: UU Nomor 6 Tahun 2023, Halaman: 1072
* Dokumen: UU Nomor 6 Tahun 2023, Halaman: 638

🧑‍💻 Anda: Apa saja syarat pembagian harta gono-gini jika terjadi perceraian menurut Kompilasi Hukum Islam?
🤖 Sistem sedang mencari referensi...
=> GUARDRAIL AKTIF: Kata ['syarat', 'pembagian', 'gono', 'gini', 'terjadi', 'perceraian', 'kompilasi', '

In [ ]:
# Sel Tambahan: Generate Exact Version requirements.txt

print("Mendeteksi versi asli library di environment saat ini...\n")

# DaftaR menyusun arsitektur Qwen, RAG, dan Tools
core_packages = [
    # Core AI & LLM
    "torch",
    "transformers",
    "accelerate",
    "bitsandbytes",
    "peft",
    "trl",

    # RAG Ecosystem (LangChain)
    "langchain",
    "langchain-community",
    "langchain-core",
    "langchain-text-splitters",

    # Vector Database & Retrieval
    "chromadb",
    "sentence-transformers",
    "rank_bm25",

    # Tools & Fallback
    "duckduckgo-search",

    # Data & Utility
    "numpy",
    "pandas"
]

requirements_lines = ["# Versi Exact dari Environment Qwen Hybrid RAG\n"]

for pkg in core_packages:
    try:
        # Mengambil versi persis yang saat ini ter-install dan berjalan
        version = importlib.metadata.version(pkg)
        line = f"{pkg}=={version}"
        requirements_lines.append(line)
        print(f"[OK] {line}")
    except importlib.metadata.PackageNotFoundError:
        print(f"[WARNING] {pkg} tidak terdeteksi (mungkin sudah build-in atau ada perbedaan nama modul).")

# Khusus Unsloth, kita tetap menggunakan format GitHub karena instalasinya khusus Colab
requirements_lines.append("\n# Unsloth (Instalasi khusus via GitHub untuk Colab)")
requirements_lines.append("unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git")

# Menyimpan ke dalam file text
filename = "requirements_exact.txt"
with open(filename, "w") as f:
    f.write("\n".join(requirements_lines))

# Memicu unduhan otomatis
files.download(filename)
print(f"\n[SUCCESS] File {filename} dengan versi ASLI berhasil dibuat dan diunduh!")

Mendeteksi versi asli library di environment saat ini...

[OK] torch==2.11.0+cu128
[OK] transformers==5.5.0
[OK] accelerate==1.13.0
[OK] bitsandbytes==0.49.2
[OK] peft==0.19.1
[OK] trl==0.24.0
[OK] langchain==1.3.4
[OK] langchain-community==0.4.2
[OK] langchain-core==1.4.0
[OK] langchain-text-splitters==1.1.2
[OK] chromadb==1.5.9
[OK] sentence-transformers==5.5.1
[OK] rank_bm25==0.2.2
[WARNING] duckduckgo-search tidak terdeteksi (mungkin sudah build-in atau ada perbedaan nama modul).
[OK] numpy==2.0.2
[OK] pandas==2.2.2


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


[SUCCESS] File requirements_exact.txt dengan versi ASLI berhasil dibuat dan diunduh!
